<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase25_carry_fx_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 25: carry en divisas

Se agrega el diferencial de tasas (carry) al rendimiento de seis pares contra el dólar, se prueba una señal de carry con control de tasas repartidas al azar y se combina con el momentum de la clase 23.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

# EJEMPLO A MANO 1: cuanto cobra una posicion corta en USD/MXN
# Tasas anuales supuestas: peso 7%, dolar 4%. dif = tasa del dolar menos tasa de la otra moneda.
r_mxn, r_usd = 0.07, 0.04
dif = r_usd - r_mxn                      # -3%: el dolar paga menos, asi que conviene estar corto USD (largo peso)
pos = -1                                 # corto USD
rend_precio = np.array([0.002, -0.001, 0.004, -0.003, 0.001])   # rendimientos diarios de USD/MXN (5 dias)
pnl_precio = pos * rend_precio           # corto USD: si USD/MXN sube pierdes
pnl_carry = np.full(5, pos * dif / 252)  # el carry diario: pos * dif / 252 (positivo cuando pos y dif tienen signos opuestos)
print("Solo precio :", (pnl_precio * 100).round(3), "| total %.3f%%" % (pnl_precio.sum() * 100))
print("Solo carry  :", (pnl_carry * 100).round(4), "| total %.4f%%" % (pnl_carry.sum() * 100))
print("Precio + carry: total %.4f%%" % ((pnl_precio + pnl_carry).sum() * 100))

# EJEMPLO A MANO 2: cuanto pesa el carry frente al riesgo del precio
# Un carry de 3% anual con una volatilidad de precio de 12% anual: si el precio no tuviera tendencia, el Sharpe seria
carry_anual, vol_precio = 0.03, 0.12
print("\nSharpe del carry puro (sin tendencia del precio): %.2f" % (carry_anual / vol_precio))
# Y si el precio se mueve en contra 1 desviacion estandar en un año (12%), el carry de 3% no alcanza a cubrirlo:
print("Resultado del año si el precio se mueve 1 desviacion en contra: %.0f%%" % ((carry_anual - vol_precio) * 100))


Solo precio : [-0.2  0.1 -0.4  0.3 -0.1] | total -0.300%
Solo carry  : [0.0119 0.0119 0.0119 0.0119 0.0119] | total 0.0595%
Precio + carry: total -0.2405%

Sharpe del carry puro (sin tendencia del precio): 0.25
Resultado del año si el precio se mueve 1 desviacion en contra: -9%


In [ ]:
# Funciones (las mismas de la clase 24, mas lo del carry)
def vol_ewma(rend, lam=0.94):
    """Volatilidad anualizada con EWMA, usando datos hasta t (inclusive)."""
    return np.sqrt((rend ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)

def backtest(rend, pos, rezago=1, costo_bps=0.0):
    """La posicion al cierre de t gana el rendimiento de t+1 (rezago=1). 'rend' es el rendimiento de estar largo USD."""
    pos_usada = pos.shift(rezago)
    bruto = (pos_usada * rend).mean(axis=1)
    rotacion = pos_usada.diff().abs().mean(axis=1).fillna(0.0)
    return bruto - rotacion * costo_bps / 10000, rotacion

def sharpe(x):
    x = x.dropna()
    return x.mean() / x.std() * np.sqrt(252)

def metricas(pnl):
    pnl = pnl.dropna()
    anios = len(pnl) / 252
    s = sharpe(pnl)
    t = s * np.sqrt(anios)
    return {"rend_%": pnl.mean() * 252 * 100, "vol_%": pnl.std() * np.sqrt(252) * 100, "Sharpe": s,
            "t": t, "p": 2 * (1 - norm.cdf(abs(t)))}

def caida_maxima(pnl):
    c = pnl.dropna().cumsum()
    return (c - c.cummax()).min() * 100


In [ ]:
# Datos: precios de seis pares contra el dolar (como en la clase 24) y tasas de interes de corto plazo (FRED, mensuales)
pares = {"USD/MXN": ("MXN=X", False, "MXN"), "USD/JPY": ("JPY=X", False, "JPY"), "USD/EUR": ("EURUSD=X", True, "EUR"),
         "USD/GBP": ("GBPUSD=X", True, "GBP"), "USD/CAD": ("CAD=X", False, "CAD"), "USD/BRL": ("BRL=X", False, "BRL")}
px = {}
for nombre, (ticker, invertir, _) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False, auto_adjust=True)["Close"].squeeze()
    px[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(px).dropna().pct_change().dropna()     # rendimiento de estar largo USD (precio en moneda extranjera por dolar)
CALENTAMIENTO = 505                                        # las mismas fechas de evaluacion de la clase 24
print("Dias:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date())

# Tasas de corto plazo de FRED (una por mes). Para cada moneda se prueban varias series, en este orden, y se usa la primera que baje y este al dia.
ids_tasa = {"USD": ["IR3TIB01USM156N", "IRSTCI01USM156N", "FEDFUNDS"],
            "MXN": ["IR3TIB01MXM156N", "IRSTCI01MXM156N", "INTDSRMXM193N"],
            "JPY": ["IR3TIB01JPM156N", "IRSTCI01JPM156N", "INTDSRJPM193N"],
            "EUR": ["IR3TIB01EZM156N", "IRSTCI01EZM156N", "ECBDFR"],
            "GBP": ["IR3TIB01GBM156N", "IRSTCI01GBM156N", "INTDSRGBM193N"],
            "CAD": ["IR3TIB01CAM156N", "IRSTCI01CAM156N", "INTDSRCAM193N"],
            "BRL": ["IRSTCI01BRM156N", "INTDSRBRM193N", "IR3TIB01BRM156N"]}
def tasa_fred(serie):
    s = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={serie}", index_col=0, parse_dates=True).iloc[:, 0]
    s = pd.to_numeric(s, errors="coerce").dropna()
    return s.resample("MS").mean().dropna()                # una observacion por mes (promedio), tambien para series diarias como ECBDFR
limite = rend.index[-1] - pd.DateOffset(months=6)          # una serie que termina antes de esta fecha se considera desactualizada
tasas, usada = {}, {}
for moneda, lista in ids_tasa.items():
    bajadas = {}
    for sid in lista:
        try:
            bajadas[sid] = tasa_fred(sid)
        except Exception as e:
            print("   no sirvio %s (%s): %s" % (moneda, sid, str(e)[:60]))
    if not bajadas:
        continue
    al_dia = [sid for sid in bajadas if bajadas[sid].index[-1] >= limite]
    elegida = al_dia[0] if al_dia else max(bajadas, key=lambda k: bajadas[k].index[-1])
    tasas[moneda], usada[moneda] = bajadas[elegida], elegida
    if not al_dia:
        print("   AVISO: ninguna serie de %s esta al dia; se usa %s" % (moneda, elegida))
faltan = [m for m in ids_tasa if m not in tasas]
if faltan:
    print("SIN TASA para:", faltan, "-> se quitan esos pares del analisis (ya no son seis, y eso cambia la comparacion con la clase 24)")
    pares = {k: v for k, v in pares.items() if v[2] in tasas}
    rend = rend[list(pares)]
if "USD" not in tasas:
    raise SystemExit("Sin tasa del dolar. Mandame este mensaje.")
for m, s in tasas.items():
    print("%s (%s): ultima fecha %s | ultimo valor %.2f%% | promedio desde 2020-10: %.2f%%" % (m, usada[m], s.index[-1].date(), s.iloc[-1], s.loc["2020-10-01":].mean()))

# Sin mirar el futuro: la tasa del mes m (publicada despues) solo se usa a partir del mes m+1
tasas_df = pd.DataFrame(tasas)
tasas_df.index = tasas_df.index + pd.DateOffset(months=1)
tasas_d = tasas_df.reindex(tasas_df.index.union(rend.index)).ffill().reindex(rend.index)
for m, s in tasas.items():
    if s.index[-1] + pd.DateOffset(months=1) < rend.index[-1]:
        print("AVISO: la tasa de %s se congela desde %s (ultimo dato de FRED)" % (m, s.index[-1].date()))

# Diferencial anual: tasa del dolar menos tasa de la otra moneda (positivo: estar largo USD cobra intereses)
dif = pd.DataFrame({p: (tasas_d["USD"] - tasas_d[c]) / 100 for p, (_, _, c) in pares.items()})
rend_int = dif / 252                                       # intereses diarios de estar largo USD
rend_tot = rend + rend_int                                 # rendimiento total de estar largo USD
print("\nDiferencial promedio (USD menos la otra moneda, en %), del 2022-09-15 al final:")
print((dif.iloc[CALENTAMIENTO:].mean() * 100).round(2).to_string())


Dias: 1570 | del 2020-10-08 al 2026-10-21
   no sirvio BRL (IR3TIB01BRM156N): HTTP Error 404: Not Found
USD (IR3TIB01USM156N): ultima fecha 2026-10-01 | ultimo valor 4.12% | promedio desde 2020-10: 3.28%
MXN (IR3TIB01MXM156N): ultima fecha 2026-10-01 | ultimo valor 6.81% | promedio desde 2020-10: 8.43%
JPY (IR3TIB01JPM156N): ultima fecha 2026-09-01 | ultimo valor 1.68% | promedio desde 2020-10: 0.32%
EUR (ECBDFR): ultima fecha 2026-11-01 | ultimo valor 2.50% | promedio desde 2020-10: 1.74%
GBP (IRSTCI01GBM156N): ultima fecha 2026-10-01 | ultimo valor 3.73% | promedio desde 2020-10: 3.03%
CAD (IR3TIB01CAM156N): ultima fecha 2026-10-01 | ultimo valor 2.45% | promedio desde 2020-10: 2.61%
BRL (IRSTCI01BRM156N): ultima fecha 2026-10-01 | ultimo valor 13.75% | promedio desde 2020-10: 11.22%
AVISO: la tasa de JPY se congela desde 2026-09-01 (ultimo dato de FRED)

Diferencial promedio (USD menos la otra moneda, en %), del 2022-09-15 al final:
USD/MXN   -5.30
USD/JPY    4.05
USD/EUR    1.80
US

In [ ]:
# CARRY: largo la moneda con tasa mas alta, corto la de tasa mas baja, contra el dolar
escala = (0.10 / vol_ewma(rend)).clip(upper=3.0)
carry = np.sign(dif)                                       # +1: largo USD (el dolar paga mas); -1: corto USD
mom = np.sign(np.log1p(rend).rolling(126).sum())           # momentum de 126 dias (clase 23 y 24)
pos_comb = (carry + mom) / 2 * escala

pnls = {
    "Carry (con intereses)": backtest(rend_tot, carry * escala)[0].iloc[CALENTAMIENTO:],
    "Carry (solo precio)": backtest(rend, carry * escala)[0].iloc[CALENTAMIENTO:],
    "Momentum 126d (rend. total)": backtest(rend_tot, mom * escala)[0].iloc[CALENTAMIENTO:],
    "Combinada carry + momentum": backtest(rend_tot, pos_comb)[0].iloc[CALENTAMIENTO:],
    "Siempre largo USD (rend. total)": backtest(rend_tot, escala)[0].iloc[CALENTAMIENTO:],
}
tabla = pd.DataFrame({k: metricas(v) for k, v in pnls.items()}).T
tabla["p_Bonferroni"] = (tabla["p"] * 3).clip(upper=1.0)   # 3 pruebas: carry, momentum y combinada
tabla.loc[["Carry (solo precio)", "Siempre largo USD (rend. total)"], "p_Bonferroni"] = np.nan
print(tabla.round(3).to_string())
anios = len(pnls["Carry (con intereses)"]) / 252
print("Error estandar aproximado del Sharpe:", round(1 / np.sqrt(anios), 2), "| años:", round(anios, 2))

print("\nCorrelacion entre el carry y el momentum:", round(pnls["Carry (con intereses)"].corr(pnls["Momentum 126d (rend. total)"]), 2))
print("\nFraccion de dias largo USD con la señal de carry:")
print((carry.iloc[CALENTAMIENTO:] > 0).mean().round(2).to_string())


                                 rend_%  vol_%  Sharpe      t      p  p_Bonferroni
Carry (con intereses)             4.372  4.669   0.936  1.925  0.054         0.163
Carry (solo precio)               0.634  4.666   0.136  0.280  0.780           NaN
Momentum 126d (rend. total)      -3.425  5.725  -0.598 -1.230  0.219         0.656
Combinada carry + momentum        0.473  3.662   0.129  0.266  0.790         1.000
Siempre largo USD (rend. total)   0.525  7.477   0.070  0.144  0.885           NaN
Error estandar aproximado del Sharpe: 0.49 | años: 4.23

Correlacion entre el carry y el momentum: -0.02

Fraccion de dias largo USD con la señal de carry:
USD/MXN    0.00
USD/JPY    1.00
USD/EUR    1.00
USD/GBP    0.72
USD/CAD    0.97
USD/BRL    0.00


In [ ]:
# De donde viene el rendimiento del carry: intereses contra movimiento del precio, y su riesgo de cola
pos_c = carry * escala
precio = backtest(rend, pos_c)[0].iloc[CALENTAMIENTO:]
interes = backtest(rend_int, pos_c)[0].iloc[CALENTAMIENTO:]
total = precio + interes
print("Rendimiento anual del carry: precio %.2f%% | intereses %.2f%% | total %.2f%%" % (
    precio.mean() * 252 * 100, interes.mean() * 252 * 100, total.mean() * 252 * 100))

mensual = pd.DataFrame({k: v.resample("ME").sum() * 100 for k, v in pnls.items()
                        if k in ("Carry (con intereses)", "Combinada carry + momentum", "Siempre largo USD (rend. total)")})
resumen = pd.DataFrame({"peor mes %": mensual.min(), "mejor mes %": mensual.max(), "asimetria": mensual.skew(),
                        "caida maxima %": pd.Series({k: caida_maxima(pnls[k]) for k in mensual.columns})})
print("\nRiesgo de cola (rendimientos mensuales):")
print(resumen.round(2).to_string())
print("Peor mes del carry:", mensual["Carry (con intereses)"].idxmin().date())


Rendimiento anual del carry: precio 0.63% | intereses 3.74% | total 4.37%

Riesgo de cola (rendimientos mensuales):
                                 peor mes %  mejor mes %  asimetria  caida maxima %
Carry (con intereses)                 -3.74         2.48      -0.77           -7.43
Combinada carry + momentum            -2.96         2.29      -0.50           -8.91
Siempre largo USD (rend. total)       -4.53         4.91       0.25          -13.03
Peor mes del carry: 2024-08-31


In [ ]:
# CONTROL: ¿importa que cada par tenga la señal de carry que le toca? Se reparten las SEÑALES (no los intereses) entre pares al azar, 200 veces.
# Los rendimientos y los intereses que se cobran son siempre los reales (rend_tot); solo cambia que posicion (largo o corto USD) lleva cada par.
rng = np.random.default_rng(11)
cols = list(rend.columns)
real = sharpe(backtest(rend_tot, carry * escala)[0].iloc[CALENTAMIENTO:])
sims = []
while len(sims) < 200:
    perm = rng.permutation(len(cols))
    if (perm == np.arange(len(cols))).all():
        continue
    carry_p = carry.iloc[:, perm].copy()
    carry_p.columns = cols
    sims.append(sharpe(backtest(rend_tot, carry_p * escala)[0].iloc[CALENTAMIENTO:]))
sims = np.array(sims)
print("Sharpe real del carry: %.2f" % real)
print("Con señales repartidas al azar: promedio %.2f | desv. est. %.2f | minimo %.2f | maximo %.2f" % (sims.mean(), sims.std(), sims.min(), sims.max()))
print("Fraccion de repartos al azar con Sharpe igual o mayor al real: %.3f" % (sims >= real).mean())


Sharpe real del carry: 0.94
Con señales repartidas al azar: promedio -0.03 | desv. est. 0.57 | minimo -1.31 | maximo 0.94
Fraccion de repartos al azar con Sharpe igual o mayor al real: 0.005


In [ ]:
# Costos (puntos base por unidad operada: SUPUESTO mio, cambialo por el de tu mesa)
filas = []
for nombre, pos in [("Carry", carry * escala), ("Momentum 126d", mom * escala), ("Combinada carry + momentum", pos_comb)]:
    fila = {"estrategia": nombre, "rotacion_anual": backtest(rend_tot, pos, 1)[1].iloc[CALENTAMIENTO:].mean() * 252}
    for bps in (0, 1, 2, 5):
        fila[f"Sharpe {bps}bps"] = sharpe(backtest(rend_tot, pos, 1, bps)[0].iloc[CALENTAMIENTO:])
    filas.append(fila)
print(pd.DataFrame(filas).round(2).to_string(index=False))


                estrategia  rotacion_anual  Sharpe 0bps  Sharpe 1bps  Sharpe 2bps  Sharpe 5bps
                     Carry           11.49         0.94         0.91         0.89         0.81
             Momentum 126d           44.69        -0.60        -0.68        -0.75        -0.99
Combinada carry + momentum           23.51         0.13         0.07         0.00        -0.19
